# Multidimensional feature tuning in category‑selective areas of human visual cortex

In [ ]:
from scripts.run_prepro import run_prepro
from scripts.run_bcv import run_bcv
from scripts.run_bnmf import run_bnmf
from scripts.run_interpret import run_interpret
from scripts.run_encoding import run_encoding

from roidims.bnmf import compute_evar_consistent, compute_sim_dims_groups
from roidims.encoding import compute_corr_H_beta, selectivity_vs_sparseness

from roidims.plotting import (
    fig_consistency,
    fig_top_imgs,
    fig_dprime_vs_beta,
    fig_wordclouds,
    fig_flatmaps_encoding_r2,
    fig_r2_bar,
    fig_flatmaps_encoding_betas,
    suppfig_k_optim,
    suppfig_consistency,
    suppfig_sim_dims,
    suppfig_sparseness,
    suppfig_noiseceilings,
)

## Background
What functional organization underlies both category-selective areas and continuous feature maps in high-level visual cortex?

We addressed this question by applying a data-driven decomposition to fMRI responses from face-, body-, and scene-selective areas, revealing that both views reflect a common principle: multidimensional tuning that is clustered within areas yet distributed across cortex.

## fMRI data
We used a subset of the Natural Scenes Dataset (NSD; Allen et al., 2022) including fMRI responses (7T) from four participants, each viewing 10,000 natural images.

We defined FFA, PPA, and EBA using an independent functional localizer (fLoc; Stigliani et al., 2015), selecting voxels with a strong category preference (t > 2) and sufficient reliability (SNR > 0.2).

In [ ]:
# Specify subjetcs and ROIs
subjects = ["subj01", "subj02", "subj05", "subj07"]
rois = ["FFA", "EBA", "PPA"]

## 0. Preprocessing
Single-trial responses were preprocessed with GLMdenoise and ridge regression, z-scored within sessions, averaged across repetitions, and split 70/30 into training and test sets.

In [ ]:
# Preprocessing
run_prepro()

## 1. Data-driven voxel decomposition
First, we applied non-negative matrix factorization to reveal which underlying representational dimensions capture the fMRI activity patterns.

### Background: Bayesian non-negative matrix factorization (BNMF)
We decomposed voxel responses from each ROI into representational dimensions using BNMF (Schmidt et al., 2009; following Khosla et al., 2022).

BNMF factorizes the image × voxel response matrix into a response matrix W (image × dimension) and a weight matrix H (dimension × voxel). The non-negativity constraint encourages sparse, additive, part-based dimensions, allowing voxels to participate in multiple dimensions simultaneously. The Bayesian extension adds exponential priors on W and H and uses Gibbs sampling to obtain robust posterior estimates.

We ran 3,000 iterations (1,000 burn-in, every 5th sample retained) and baseline-shifted responses to enforce non-negativity, using the training minimum for both sets to prevent data leakage.

### 1.1. Estimating the optimal dimensionality
We determined the optimal number of dimensions k* for each participant and ROI using bi-cross-validation (Owen & Perry, 2009).

Bi-cross-validation partitions the data into four blocks. BNMF is trained on one block, and pseudo-inverse operations on two others are used to predict the held-out block. This was repeated 5 times across ranks 1–100 (step size 3), and k* was chosen as the rank minimizing average test error.

In [ ]:
# Bi-cross-validation
run_bcv(subjects, rois, k_min=1, k_max=100, k_steps=1, n_perms=5)

In [ ]:
# Supp Fig 1: Optimal dimensionality
suppfig_k_optim(subjects, rois)

### 1.2. Finding reliable dimensions
To obtain reliable and generalizable dimensions, we used a two-step consensus approach:

First, we ran BNMF 100 times with k* but random initializations per ROI, removed outlier runs, and aggregated stable dimensions via k-medoids clustering (Kotliar et al., 2019).

Second, we matched dimensions across participants using pairwise correlations and a greedy selection procedure, retaining only dimensions consistent across all four participants (r > 0.3; Khosla et al., 2022).

This yielded 8–20 consensus dimensions per ROI. To test the generalizability to unseen images, we projected the test set into the learned embedding using non-negative least squares regression.

In [ ]:
# Specify inter-subject consistency threshold
r_thresh = 0.3

In [ ]:
# Consensus approach
run_bnmf(subjects, rois, n_runs=100, r_thresh=r_thresh)

In [ ]:
# Compute explained variance
compute_evar_consistent(subjects, rois)

In [ ]:
# Fig 2a: Consistency
fig_consistency(rois, r_thresh)

In [ ]:
# Supp Fig 2: Mean inter-participant consistency of all dimensions
suppfig_consistency(rois, r_thresh)

In [ ]:
# Fig 2b: Top images combined across subjects
fig_top_imgs(subjects, rois)

In [ ]:
# Supp Fig 3: Mean similarity matrices
suppfig_sim_dims(subjects, rois)

In [ ]:
# Representational similarity of all dimensions
compute_sim_dims_groups(subjects, rois)

## 2. Data-driven dimension labeling
To interpret each dimension, we used a behavioral online experiment. We collected data from N=38 participants, who viewed collages of the 20 highest-scoring images per dimension and provided 3–5 short labels describing what the images had in common. Labels were translated from German, cleaned of generic entries, and standardized.

We then used CLIP-ViT-L/14 (Radford et al., 2021) to select the most representative label per dimension from the participant-generated candidate set. For each label, we computed the cosine similarity between its text embedding and all image embeddings, weighted by each dimension's response profile, then z-scored across labels within each dimension to highlight selectively associated labels. Top-ranked labels were visualized as word clouds, yielding concise, semantically meaningful descriptors validated across the full image set.

### 2.1. Finding the most representative labels

In [ ]:
# CLIP-based interpretation
run_interpret(subjects, rois)

In [ ]:
# Fig 3: Top labels for all subjects
fig_wordclouds(subjects, rois)

## 3. Testing the relationship between multidimensional tuning and category selectivity
To quantify category selectivity for the ROI's preferred category, we used a category selectivity index (d'), comparing mean responses to a preferred category against all others, weighted by response variance.

To measure the extent of multidimensional tuning, we computed a sparseness index (Hoyer, 2004) based on the normalized L1/L2-norm ratio of each voxel's dimension weight profile, ranging from 0 (tuned equally to all dimensions) to 1 (tuned to a single dimension).

In [ ]:
# Fig 3: Correlation between dimension tuning (a.u.) and category selectivity (d')
fig_dprime_vs_beta(subjects, rois)

In [ ]:
# Plot sparseness for different selectivity bins
suppfig_sparseness(subjects, rois)

In [ ]:
# Print correlation
selectivity_vs_sparseness(subjects, rois)

## 4. Voxel-wise encoding models
Finally, we fit voxel-wise encoding models to predict fMRI responses to held-out images from the learned dimensions, revealing the cortical topography of this organization.

We used fractional ridge regression with cross-validation to avoid overfitting, projected test images into the learned BNMF space with non-negative least squares, and evaluated prediction performance on held-out data. Significance was assessed with permutation tests and FDR correction, and noise ceilings were computed to benchmark model performance.

In [ ]:
# Fractional ridge regression
run_encoding(subjects, rois, n_folds=10, n_perms=3000)

In [ ]:
# Print correlation between ROI coefficient weights
compute_corr_H_beta(subjects, rois)

In [ ]:
# Fig 4a: Prediction performance maps
fig_flatmaps_encoding_r2

In [ ]:
# Fig 4b: Prediction performance across ROIs
fig_r2_bar(subjects, rois)

In [ ]:
# Supp Fig 6: Voxel-wise prediction performance vs. noise ceiling estimate
suppfig_noiseceilings(subjects, rois)

In [ ]:
# Fig 5: Individual dimension tuning maps
fig_flatmaps_encoding_betas(subjects, rois)